In [7]:
# Cell 1: Imports and Setup
import psycopg2
import psycopg2.extensions
import select
import json
import logging
import os
from datetime import datetime
from typing import Dict, Any
from pathlib import Path
import yaml
import pandas as pd
import time
import sys

# Get the absolute path to the project root
notebook_path = Path().absolute()
project_root = notebook_path.parent.parent  # Adjust this based on your actual directory structure

# Add the project root to Python path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Print paths for debugging
print("Current directory:", notebook_path)
print("Project root:", project_root)
print("Python path:", sys.path)

# Alternative path setup
notebook_path = Path().absolute()
project_root = notebook_path.parent  # Only one parent up to reach src/
sys.path.append(str(project_root))

# Now the import should work
from indicator.indicator import Indicator


Current directory: c:\My Folder\CryptoAPI\src\listener
Project root: c:\My Folder\CryptoAPI
Python path: ['c:\\Users\\micha\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'c:\\Users\\micha\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'c:\\Users\\micha\\AppData\\Local\\Programs\\Python\\Python312\\Lib', 'c:\\Users\\micha\\AppData\\Local\\Programs\\Python\\Python312', '', 'C:\\Users\\micha\\AppData\\Roaming\\Python\\Python312\\site-packages', 'C:\\Users\\micha\\AppData\\Roaming\\Python\\Python312\\site-packages\\win32', 'C:\\Users\\micha\\AppData\\Roaming\\Python\\Python312\\site-packages\\win32\\lib', 'C:\\Users\\micha\\AppData\\Roaming\\Python\\Python312\\site-packages\\Pythonwin', 'c:\\Users\\micha\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages', '.', '.', 'c:\\My Folder\\CryptoAPI', 'c:\\My Folder\\CryptoAPI\\src']


In [8]:
# Cell 2: Logging Setup
log_dir = Path().parent.parent / 'logs'
log_dir.mkdir(exist_ok=True)
log_file = log_dir / 'kline_listener.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [9]:
# Cell 3: KlineListener Class Definition
class KlineListener:
    def __init__(self, source_db_params: Dict[str, str], target_db_params: Dict[str, str]):
        self.source_db_params = source_db_params
        self.target_db_params = target_db_params
        self.source_conn = None
        self.target_conn = None
    
    def connect(self) -> None:
        try:
            self.source_conn = psycopg2.connect(**self.source_db_params)
            self.source_conn.set_isolation_level(psycopg2.extensions.ISOLATION_LEVEL_AUTOCOMMIT)
            
            self.target_conn = psycopg2.connect(**self.target_db_params)
            self.target_conn.autocommit = True
            
            logger.info("Successfully connected to both databases")
        except Exception as e:
            logger.error(f"Error connecting to databases: {str(e)}")
            raise

In [11]:
# Cell 4: Table Management Methods
def _ensure_indicator_table(self, table_name: str) -> None:
        with self.target_conn.cursor() as cur:
            cur.execute(f"""
                CREATE TABLE IF NOT EXISTS {table_name} (
                    id SERIAL PRIMARY KEY,
                    source_id INTEGER UNIQUE NOT NULL,
                    ts TIMESTAMP NOT NULL,
                    open NUMERIC(20, 8),
                    high NUMERIC(20, 8),
                    low NUMERIC(20, 8),
                    close NUMERIC(20, 8),
                    ema_14 NUMERIC(20, 8),
                    rsi_14 NUMERIC(20, 8),
                    bb_upper NUMERIC(20, 8),
                    bb_middle NUMERIC(20, 8),
                    bb_lower NUMERIC(20, 8),
                    stoch_rsi NUMERIC(20, 8),
                    stoch_k NUMERIC(20, 8),
                    stoch_d NUMERIC(20, 8),
                    atr NUMERIC(20, 8),
                    cci NUMERIC(20, 8),
                    roc_12 NUMERIC(20, 8),
                    momentum_14 NUMERIC(20, 8),
                    psar NUMERIC(20, 8),
                    williams_r_14 NUMERIC(20, 8),
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)
            self.target_conn.commit()

In [13]:
# Cell 4: Table Management Methods
def _ensure_indicator_table(self, table_name: str) -> None:
        with self.target_conn.cursor() as cur:
            cur.execute(f"""
                CREATE TABLE IF NOT EXISTS {table_name} (
                    id SERIAL PRIMARY KEY,
                    source_id INTEGER UNIQUE NOT NULL,
                    ts TIMESTAMP NOT NULL,
                    open NUMERIC(20, 8),
                    high NUMERIC(20, 8),
                    low NUMERIC(20, 8),
                    close NUMERIC(20, 8),
                    ema_14 NUMERIC(20, 8),
                    rsi_14 NUMERIC(20, 8),
                    bb_upper NUMERIC(20, 8),
                    bb_middle NUMERIC(20, 8),
                    bb_lower NUMERIC(20, 8),
                    stoch_rsi NUMERIC(20, 8),
                    stoch_k NUMERIC(20, 8),
                    stoch_d NUMERIC(20, 8),
                    atr NUMERIC(20, 8),
                    cci NUMERIC(20, 8),
                    roc_12 NUMERIC(20, 8),
                    momentum_14 NUMERIC(20, 8),
                    psar NUMERIC(20, 8),
                    williams_r_14 NUMERIC(20, 8),
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)
            self.target_conn.commit()

In [16]:
# Cell 6: Process Notification Method
def process_notification(self, payload: Dict[str, Any]) -> None:
        """Process notification and calculate indicators"""
        try:
            # Extract data from payload
            source_table = payload['table']
            symbol = payload['symbol']
            timeframe = self._normalize_timeframe(payload['timeframe'])
            
            # Construct indicator table name
            indicator_table = f"{source_table}_indicators"
            
            logger.info(f"Processing {symbol} {timeframe} data update")
            
            # Ensure table exists
            self._ensure_indicator_table(indicator_table)
            
            # Get historical data for calculations
            with self.source_conn.cursor() as source_cur:
                source_cur.execute(f"""
                    SELECT id, ts, 
                        CAST(open AS FLOAT), CAST(high AS FLOAT), 
                        CAST(low AS FLOAT), CAST(close AS FLOAT)
                    FROM {source_table}
                    WHERE ts <= %s
                    ORDER BY ts DESC
                    LIMIT 50
                """, (payload['data']['ts'],))
                
                historical_data = source_cur.fetchall()
                
                if len(historical_data) < 20:
                    logger.info(f"Insufficient historical data ({len(historical_data)} rows). Skipping.")
                    return
                
                # Create DataFrame with historical data (reverse to get ascending order)
                df = pd.DataFrame(historical_data[::-1], columns=['id', 'ts', 'open', 'high', 'low', 'close'])
                df['ts'] = pd.to_datetime(df['ts'])
                
                # Ensure numeric columns are float
                numeric_columns = ['open', 'high', 'low', 'close']
                df[numeric_columns] = df[numeric_columns].astype(float)
                
                # Calculate indicators
                indicator = Indicator(df, symbol=symbol, timeframe=timeframe)
                
                try:
                    indicator.ema(period=14)
                    indicator.rsi(period=14)
                    indicator.bollinger_bands()
                    indicator.stoch_rsi()
                    indicator.atr()
                    indicator.cci()
                    indicator.roc(period=12)
                    indicator.momentum(period=14)
                    indicator.parabolic_sar()
                    indicator.williams_r(period=14)
                except Exception as e:
                    logger.error(f"Error calculating indicators: {str(e)}")
                    return
                
                # Get the last row for insertion (most recent)
                results_df = indicator.df.iloc[-1:].copy()
                
                # Insert the calculated row
                with self.target_conn.cursor() as target_cur:
                    row = results_df.iloc[0]
                    target_cur.execute(f"""
                        INSERT INTO {indicator_table} (
                            source_id, ts, open, high, low, close,
                            ema_14, rsi_14, bb_upper, bb_middle, bb_lower,
                            stoch_rsi, stoch_k, stoch_d, atr, cci,
                            roc_12, momentum_14, psar, williams_r_14
                        ) VALUES (
                            %s, %s, %s, %s, %s, %s,
                            %s, %s, %s, %s, %s,
                            %s, %s, %s, %s, %s,
                            %s, %s, %s, %s
                        )
                        ON CONFLICT (source_id) DO UPDATE SET
                            ts = EXCLUDED.ts,
                            open = EXCLUDED.open,
                            high = EXCLUDED.high,
                            low = EXCLUDED.low,
                            close = EXCLUDED.close,
                            ema_14 = EXCLUDED.ema_14,
                            rsi_14 = EXCLUDED.rsi_14,
                            bb_upper = EXCLUDED.bb_upper,
                            bb_middle = EXCLUDED.bb_middle,
                            bb_lower = EXCLUDED.bb_lower,
                            stoch_rsi = EXCLUDED.stoch_rsi,
                            stoch_k = EXCLUDED.stoch_k,
                            stoch_d = EXCLUDED.stoch_d,
                            atr = EXCLUDED.atr,
                            cci = EXCLUDED.cci,
                            roc_12 = EXCLUDED.roc_12,
                            momentum_14 = EXCLUDED.momentum_14,
                            psar = EXCLUDED.psar,
                            williams_r_14 = EXCLUDED.williams_r_14
                    """, (
                        int(row['id']), row['ts'], 
                        float(row['open']), float(row['high']), 
                        float(row['low']), float(row['close']),
                        float(row.get('ema_14', 0) or 0), 
                        float(row.get('rsi_14', 0) or 0),
                        float(row.get('bb_upper', 0) or 0), 
                        float(row.get('bb_middle', 0) or 0), 
                        float(row.get('bb_lower', 0) or 0),
                        float(row.get('stoch_rsi', 0) or 0), 
                        float(row.get('stoch_k', 0) or 0), 
                        float(row.get('stoch_d', 0) or 0),
                        float(row.get('atr', 0) or 0), 
                        float(row.get('cci', 0) or 0),
                        float(row.get('roc_12', 0) or 0), 
                        float(row.get('momentum_14', 0) or 0),
                        float(row.get('psar', 0) or 0), 
                        float(row.get('williams_r_14', 0) or 0)
                    ))
                    
                self.target_conn.commit()
                logger.info(f"Successfully processed new row for {symbol} {timeframe}")
                
        except Exception as e:
            logger.error(f"Error processing notification: {str(e)}")
            logger.error(f"Payload: {payload}")
            self.target_conn.rollback()

In [18]:
# Cell 7: Listen Method
def listen(self) -> None:
        try:
            with self.source_conn.cursor() as cur:
                cur.execute("LISTEN kline_updates;")
                logger.info("Started listening for kline updates...")
                
                while True:
                    if select.select([self.source_conn], [], [], 5) != ([], [], []):
                        self.source_conn.poll()
                        while self.source_conn.notifies:
                            notify = self.source_conn.notifies.pop(0)
                            payload = json.loads(notify.payload)
                            self.process_notification(payload)
                            
        except Exception as e:
            logger.error(f"Error in listener: {str(e)}")
            raise
        finally:
            self.cleanup()

In [21]:
# Cell 8: Testing Setup
# Load configurations from src/config
notebook_path = Path(r"c:\My Folder\CryptoAPI\src\listener\listener_testing.ipynb")
config_dir = notebook_path.parent.parent / 'config'  # Go up to src/ then to config/

print(f"Looking for config files in: {config_dir}")

try:
    with open(config_dir / 'database_config.yaml', 'r') as f:
        source_params = yaml.safe_load(f)
        print("Successfully loaded database_config.yaml")
    
    with open(config_dir / 'indicator_config.yaml', 'r') as f:
        target_params = yaml.safe_load(f)
        print("Successfully loaded indicator_config.yaml")

    # Create listener instance
    listener = KlineListener(source_params, target_params)
    print("Created KlineListener instance")

except FileNotFoundError as e:
    print(f"Error: Could not find config file - {e}")
    print(f"Please ensure config files exist in: {config_dir}")

Looking for config files in: c:\My Folder\CryptoAPI\src\config
Successfully loaded database_config.yaml
Successfully loaded indicator_config.yaml
Created KlineListener instance


In [22]:
# Cell 9: Test Connection
listener.connect()

2024-11-27 14:37:23,304 - INFO - Successfully connected to both databases


In [24]:
def _normalize_timeframe(self, timeframe: str) -> str:
        """Convert timeframe to standard format"""
        # Map common variations to standard format
        timeframe_map = {
            '1': '1m',
            '3': '3m',
            '5': '5m',
            '15': '15m',
            '30': '30m',
            '60': '1h',
            '60m': '1h',
            '120': '2h',
            '240': '4h',
            '360': '6h',
            '720': '12h',
            '1440': '1d'
        }
        
        # If timeframe is already in correct format, return it
        if timeframe in ['1m', '3m', '5m', '15m', '30m', '1h', '2h', '4h', '6h', '12h', '1d']:
            return timeframe
        
        # Try to get from map, otherwise append 'm'
        return timeframe_map.get(timeframe, f"{timeframe}m")

In [25]:
# Cell 10: Test Timeframe Normalization
test_timeframes = ['1', '60', '240', '1m', '1h', '4h']
for tf in test_timeframes:
    print(f"{tf} -> {listener._normalize_timeframe(tf)}")

AttributeError: 'KlineListener' object has no attribute '_normalize_timeframe'